In [2]:
# db_handler.py
import sqlite3
from datetime import datetime, timedelta, timezone

DB_NAME = "news_ai.db"

In [10]:
# Initialize database and table
def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS news (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            link TEXT UNIQUE,
            date TEXT,
            source TEXT,
            category TEXT,
            fetched_at       
        )
    ''')
    conn.commit()
    conn.close()

In [ ]:
import feedparser
from datetime import datetime, timedelta, timezone
import requests

# RSS feeds organized by type
rss_feeds = {
    "Top Stories": [
        "https://www.cbc.ca/cmlink/rss-topstories",
        "https://www.thestar.com/search/?f=rss&t=article&bl=2827101&l=20",
        "https://globalnews.ca/feed/",
        "https://nationalpost.com/feed/",
    ],
    "World News": [
        "https://www.cbc.ca/cmlink/rss-world",
        "https://globalnews.ca/world/feed/",
        "http://rss.cnn.com/rss/edition_world.rss"
    ],
    "Canada News": [
        "https://www.cbc.ca/cmlink/rss-canada",
        "https://globalnews.ca/canada/feed/"
    ]
}

# Time setup for 24-hour filtering
now = datetime.now(timezone.utc)
print(now)
print(timedelta(days=1))
last_24hrs = now - timedelta(days=1)
print(last_24hrs)

# Final result list
all_news = []
print(rss_feeds.items())

2025-03-30 08:10:18.156375+00:00
1 day, 0:00:00
2025-03-29 08:10:18.156375+00:00
dict_items([('Top Stories', ['https://www.cbc.ca/cmlink/rss-topstories', 'https://www.thestar.com/search/?f=rss&t=article&bl=2827101&l=20', 'https://globalnews.ca/feed/', 'https://nationalpost.com/feed/']), ('World News', ['https://www.cbc.ca/cmlink/rss-world', 'https://globalnews.ca/world/feed/', 'http://rss.cnn.com/rss/edition_world.rss']), ('Canada News', ['https://www.cbc.ca/cmlink/rss-canada', 'https://globalnews.ca/canada/feed/'])])


In [5]:
# fetch_rss.py (with timeout + fallback)
def news():
    rss_feeds_news = []
    for urls in rss_feeds.values():
            for url in urls:
                try:
                    # Fetch RSS with timeout
                    response = requests.get(url, timeout=5)
                    feed = feedparser.parse(response.content)

                    for entry in feed.entries:
                        # print(entry)
                        try:
                            published_time = datetime(*entry.published_parsed[:6])
                            published_time = published_time.replace(tzinfo=timezone.utc)
                            # print(published_time)
                            if published_time > last_24hrs:
                                # print(entry.title)
                                news_item = {
                                    "title": entry.title,
                                    "link": entry.link,
                                    "date": published_time.strftime('%Y-%m-%d %H:%M:%S'), #todo
                                    "source": feed.feed.get("title", "Unknown Source"), #todo
                                }
                                rss_feeds_news.append(news_item)
                        except Exception as e:
                            print(f"⚠️ Entry error: {e}")
                            continue

                except requests.exceptions.Timeout:
                    print(f"⏰ Timeout: {url}")
                except Exception as e:
                    print(f"❌ Failed to fetch {url}: {e}")

    return rss_feeds_news


In [56]:
# Insert news into database (skip duplicates)
def insert_news(news_list):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    now = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
    print("insert start")
    for news in news_list:
        try:
            cursor.execute('''
                INSERT OR IGNORE INTO news (title, link, date, source, fetched_at)
                VALUES (?, ?, ?, ?, ?)
            ''', (news["title"], news["link"], news["date"], news["source"], now))
        except Exception:
            continue
    print("insert done")
    conn.commit()
    conn.close()


# Delete news older than X hours (default: 48 hours)
def delete_old_news(hours=48):
    threshold_time = (datetime.now(timezone.utc) - timedelta(hours=hours)).strftime('%Y-%m-%d %H:%M:%S')
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM news WHERE fetched_at < ?", (threshold_time,))
    conn.commit()
    conn.close()


# Get news by optional filters (e.g., keyword only)
def get_news(keyword=None, limit=100):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    query = "SELECT title, link, date, source FROM news WHERE 1=1"
    params = []

    if keyword:
        query += " AND LOWER(title) LIKE ?"
        params.append(f"%{keyword.lower()}%")

    query += " ORDER BY date DESC LIMIT ?"
    params.append(limit)

    cursor.execute(query, params)
    results = cursor.fetchall()
    conn.close()

    return [
        {"title": r[0], "link": r[1], "date": r[2], "source": r[3]}
        for r in results
    ]


In [19]:
init_db()

In [51]:
news_list = news()

⏰ Timeout: https://www.cbc.ca/cmlink/rss-topstories
⏰ Timeout: https://www.cbc.ca/cmlink/rss-world
⚠️ Entry error: object has no attribute 'published_parsed'
⚠️ Entry error: object has no attribute 'published_parsed'
⚠️ Entry error: object has no attribute 'published_parsed'
⚠️ Entry error: object has no attribute 'published_parsed'
⚠️ Entry error: object has no attribute 'published_parsed'
⚠️ Entry error: object has no attribute 'published_parsed'
⏰ Timeout: https://www.cbc.ca/cmlink/rss-canada


In [52]:
news_list

[{'title': "Judge rejects Hudson’s Bay restructuring plan as 'neither necessary nor appropriate'",
  'link': 'https://www.thestar.com/business/judge-rejects-hudson-s-bay-restructuring-plan-as-neither-necessary-nor-appropriate/article_eacd32b2-640f-446c-80e1-aa0bf38cf41c.html',
  'date': '2025-03-29 22:45:00',
  'source': 'www.thestar.com - RSS Results of type article'},
 {'title': 'She murdered her mom at 15. She wants to become an Ontario lawyer at 37. Does this ‘bathtub girl’ deserve a second chance?',
  'link': 'https://www.thestar.com/news/ontario/she-murdered-her-mom-at-15-she-wants-to-become-an-ontario-lawyer-at-37/article_9ad7d877-400a-4e42-9ce3-a9d7cb4ef5dc.html',
  'date': '2025-03-29 09:00:00',
  'source': 'www.thestar.com - RSS Results of type article'},
 {'title': "Some former NBA players say living in Toronto was 'tough.' For Raptors veteran Garrett Temple, it's quite the opposite",
  'link': 'https://www.thestar.com/sports/raptors/some-former-nba-players-say-living-in-tor

In [57]:
insert_news(news_list=news_list)

insert start
insert done


In [54]:
get_news(keyword="canada")

[{'title': 'An ‘all-in-Canada’ supply chain? How ‘backward integration’ could work',
  'link': 'https://globalnews.ca/news/11103394/donald-trump-tariffs-canada-election-ideas/',
  'date': '2025-03-29 12:00:39',
  'source': ': Canada'},
 {'title': "Canada risks losing its elimination status for measles, Ontario's top doctor warns",
  'link': 'https://www.thestar.com/news/canada/canada-risks-losing-its-elimination-status-for-measles-ontarios-top-doctor-warns/article_66d57395-24d4-4976-a10a-53dc77b99dae.html',
  'date': '2025-03-29 11:00:00',
  'source': 'www.thestar.com - RSS Results of type article'},
 {'title': 'Year-over-year data shows more than 70% drop in flight bookings from Canada to U.S. amid Trump tariffs',
  'link': 'https://www.thestar.com/business/year-over-year-data-shows-more-than-70-drop-in-flight-bookings-from-canada-to/article_55a20a97-aabe-4728-a8ba-6d6cea441473.html',
  'date': '2025-03-29 10:00:00',
  'source': 'www.thestar.com - RSS Results of type article'},
 {'tit

In [55]:
delete_old_news()